# Preparación de Datos

A partir del EDA, una conclusión importante fue que el canal ganador cambia según la audiencia y el objetivo de campaña. Por eso, en vez de preparar una sola tabla global, aca dejamos la data lista para trabajar por contexto.


## idea que sale del analisis

- no conviene tratar todo el dataset como si fuera un solo problema
- si tiene sentido crear un `contexto = audiencia + objetivo`
- ese contexto luego nos va a servir para el baseline contextual y para el multi armed bandit personalizado


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data.carga_datos import cargar_dataset
from src.data.limpieza import limpiar_dataset_publicidad
from src.data.features import crear_contexto, agregar_metricas_por_contexto_y_canal


In [2]:
df = cargar_dataset(ROOT / 'social_media_ads_filtered.csv')
df_limpio = crear_contexto(limpiar_dataset_publicidad(df))
tabla_agregada = agregar_metricas_por_contexto_y_canal(df_limpio)

print('Shape original:', df.shape)
print('Shape limpio:', df_limpio.shape)
df_limpio.head()


Shape original: (4052, 16)
Shape limpio: (4052, 17)


,Campaign_ID,Target_Audience,Campaign_Goal,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date,Company,Contexto
0,793457,Women 45-60,Market Expansion,15 Days,Facebook,0.14,500.0,5.80,Los Angeles,English,506,3017,1,Technology,2022-10-24,Tech Titans,Women 45-60 | Market Expansion
1,470283,Men 35-44,Increase Sales,15 Days,Instagram,0.08,500.0,5.39,Los Angeles,English,506,3019,9,Technology,2022-01-11,Giga Geeks,Men 35-44 | Increase Sales
2,854343,Men 45-60,Increase Sales,15 Days,Facebook,0.13,500.0,5.37,Los Angeles,English,513,3040,6,Technology,2022-03-11,Silicon Saga,Men 45-60 | Increase Sales
3,616108,All Ages,Increase Sales,15 Days,Twitter,0.02,500.0,1.48,Los Angeles,English,518,3053,10,Technology,2022-02-26,Code Crafters,All Ages | Increase Sales
4,683515,Men 35-44,Brand Awareness,15 Days,Facebook,0.04,500.0,6.28,Los Angeles,English,521,3062,9,Technology,2022-01-05,Cyber Circuit,Men 35-44 | Brand Awareness


In [3]:
print('Numero de contextos unicos:', df_limpio['Contexto'].nunique())
print('Canales por contexto, ejemplo:')
tabla_agregada[['Contexto', 'Channel_Used', 'roi_promedio', 'conversion_promedio', 'costo_promedio']].head(12)


Numero de contextos unicos: 36
Canales por contexto, ejemplo:


,Contexto,Channel_Used,roi_promedio,conversion_promedio,costo_promedio
0,All Ages | Brand Awareness,Twitter,4.288095,0.070476,7570.991429
1,All Ages | Brand Awareness,Instagram,3.545926,0.089630,8415.268519
2,All Ages | Brand Awareness,Facebook,3.473438,0.066875,7500.683437
3,All Ages | Brand Awareness,Pinterest,0.731076,0.071429,8675.841786
4,All Ages | Increase Sales,Twitter,4.176957,0.081304,6877.010000
5,All Ages | Increase Sales,Facebook,3.944091,0.091364,7353.307727
6,All Ages | Increase Sales,Instagram,3.935667,0.086000,7170.014667
7,All Ages | Increase Sales,Pinterest,0.757573,0.083824,8454.483824
8,All Ages | Market Expansion,Facebook,4.598125,0.083750,7575.532187
9,All Ages | Market Expansion,Twitter,3.815217,0.079130,7411.020000


In [4]:
ruta_procesado = ROOT / 'data' / 'processed'
ruta_procesado.mkdir(parents=True, exist_ok=True)

df_limpio.to_csv(ruta_procesado / 'social_media_ads_limpio.csv', index=False)
tabla_agregada.to_csv(ruta_procesado / 'tabla_contexto_canal.csv', index=False)

print('Archivos guardados en', ruta_procesado)


Archivos guardados en c:\Users\pedro\Documents\Pedro Tareas\CURSOS ONLINE\DMC\AI__ENGINEER\Agentes Inteligentes y Sistemas de Recomendación Adaptativos\03. Aplica\recommendation_system\data\processed


## Preparando data:

En esta etapa ya dejamos armado el nivel de análisis correcto para el problema. Lo importante aca no era solo limpiar columnas, sino preparar una estructura donde cada contexto tenga sus canales y métricas asociadas. Eso después hace mucho mas natural pasar al baseline y al bandit.
